<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/CH13_NB01_LoRA_weather_specialist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CH13 — Weather/Geo Specialist

Builds `specialist_model`, a Qwen3-0.6B fine-tuned with LoRA for weather and
geolocation tool calling. This model is the teacher for the pruning +
knowledge-distillation work in the next notebook.

Design principles:

- Each variable name means exactly one thing everywhere in the notebook.
- `specialist_model` and `base_model` are loaded in fp16 and are not
  quantized, so the comparison isolates fine-tuning from quantization effects.
- Every diagnostic function takes the model as an explicit argument instead of
  relying on a global.
- The LoRA adapter is merged into the trained model before evaluation; the
  notebook does not perform a LoRA-into-4-bit merge.

## A. Setup

In [ ]:
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    peft==0.18.0 \
    trl==0.28.0 \
    accelerate==1.12.0 \
    scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
import json
import re
import gc
import random
from collections import Counter

import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from sklearn.model_selection import train_test_split

print(torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

Tesla T4
bf16 supported: True


In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
print("Random seed set to 42")

Random seed set to 42


### Configuration

`TRAIN_MODEL` controls whether the notebook trains a new specialist or loads one
from `HF_REPO_ID`. `PUSH_TO_HUB` controls whether a newly trained specialist
and tokenizer are uploaded after training. The remaining values configure the
model, data split, LoRA adapter, and trainer.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"                    # antes: Qwen/Qwen3-1.7B
DATASET_NAME = "Salesforce/xlam-function-calling-60k"
LORA_RANK = 16                                     # antes: 32
LORA_ALPHA = 32                                    # antes: 64
LORA_DROPOUT = 0.05
TRAIN_EPOCHS = 3                                   # antes: 1
LEARNING_RATE = 2e-4
BATCH_SIZE = 4                                     # antes: 2
MAX_SEQ_LENGTH = 1024                              # antes: 1792
GRAD_ACC_STEPS = 2                                 # antes: 4
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]                            # sin cambios
TEST_SIZE = 0.30                                   # sin cambios
RANDOM_STATE = 42                                  # sin cambios
MIN_EXAMPLES_PER_FUNCTION = 3                       # sin cambios, pero ya irrelevante con 3 funciones
TRAIN_MODEL = True
PUSH_TO_HUB = True
HF_REPO_ID = "oopere/qwen3-0.6b-weather-geo-specialist"  # nuevo nombre, distinto del 1.7B

## B. Data pipeline

Curation, deduplication, stratified split, tokenizer, and prompt/completion
pairs. No model is loaded here; only the tokenizer is initialized.

In [ ]:
ALLOWED_FUNCTIONS = {"get_ip_zipcode", "get_city_from_zipcode", "local_weather_api"}


def curate_domain_dataset(dataset, allowed_functions=ALLOWED_FUNCTIONS,
                          min_examples=MIN_EXAMPLES_PER_FUNCTION):
    matched = [
        ex for ex in dataset
        if len(json.loads(ex["answers"])) == 1
        and json.loads(ex["answers"])[0]["name"] in allowed_functions
    ]
    function_counts = Counter(json.loads(ex["answers"])[0]["name"] for ex in matched)
    main_functions = {name for name, count in function_counts.items() if count >= min_examples}
    return [ex for ex in matched if json.loads(ex["answers"])[0]["name"] in main_functions]


raw_dataset = load_dataset(DATASET_NAME, split="train")
domain_examples = curate_domain_dataset(raw_dataset)

print(f"Curated domain examples: {len(domain_examples)}")
print(f"Functions covered: {len(set(json.loads(ex['answers'])[0]['name'] for ex in domain_examples))}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Curated domain examples: 233
Functions covered: 3


### Deduplication

The curated xlam data can contain repeated queries. Queries are normalized to
lowercase word text and deduplicated before the stratified train/test split, so
the evaluation set contains only queries that were held out after
deduplication.

In [ ]:
def normalize_query(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip()


def deduplicate_by_query(examples):
    seen = set()
    unique = []
    for ex in examples:
        key = normalize_query(ex["query"])
        if key not in seen:
            seen.add(key)
            unique.append(ex)
    return unique


domain_examples = deduplicate_by_query(domain_examples)
print(f"After deduplication: {len(domain_examples)}")

After deduplication: 177


### Domain split

The curated examples are filtered to the three allowed functions and to
single-answer records. After deduplication, the data is split into train and
test sets with stratification by function name. The notebook does not apply an
additional extractability filter to tool arguments.

In [ ]:
function_labels = [json.loads(ex["answers"])[0]["name"] for ex in domain_examples]

train_examples, test_examples = train_test_split(
    domain_examples, test_size=TEST_SIZE, stratify=function_labels, random_state=RANDOM_STATE,
)

print(f"Train examples: {len(train_examples)}")
print(f"Test examples: {len(test_examples)}")

Train examples: 123
Test examples: 54


### Tokenizer

Qwen3 ships a dedicated pad token (`<|endoftext|>`) distinct from its eos
(`<|im_end|>`), so there is nothing to override here. Setting `pad_token` to
`eos_token` — the reflex from older models — would make the trailing
`<|im_end|>` of every completion indistinguishable from padding. The assert
guards against that if `MODEL_NAME` is ever changed.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

assert tokenizer.pad_token_id != tokenizer.eos_token_id, \
    "pad and eos must differ, or the trailing <|im_end|> gets masked as padding"

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

pad_token: '<|endoftext|>' 151643
eos_token: '<|im_end|>' 151645


`build_prompt_completion` splits each example into `prompt` (system + tools +
user) and `completion` (only the assistant's tool_call), so training can use
`completion_only_loss=True`. Without it, most of the gradient signal goes toward
predicting the deterministic, autogenerated tool-schema JSON instead of the
actual tool-call decision.

In [ ]:
def build_prompt_completion(example, tokenizer):
    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])

    user_message = {"role": "user", "content": example["query"]}
    assistant_message = {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type": "function",
            "function": {"name": answers[0]["name"], "arguments": answers[0]["arguments"]},
        }],
    }

    full_text = tokenizer.apply_chat_template(
        [user_message, assistant_message], tools=tools, tokenize=False, enable_thinking=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        [user_message], tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    assert full_text.startswith(prompt_text), "prompt is not a prefix of the full rendered text"

    return {"prompt": prompt_text, "completion": full_text[len(prompt_text):]}


train_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in train_examples])
test_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in test_examples])

print("--- Sample prompt (tail) ---")
print(repr(train_dataset[0]["prompt"][-200:]))
print()
print("--- Sample completion ---")
print(repr(train_dataset[0]["completion"]))

--- Sample prompt (tail) ---
'\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n<|im_start|>user\nWhat city is associated with the ZIP code 94303?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

--- Sample completion ---
'<tool_call>\n{"name": "get_city_from_zipcode", "arguments": {"zipcode": "94303"}}\n</tool_call><|im_end|>\n'


#### DIAGNOSTIC: sequence length distribution

In [ ]:
lengths = [
    len(tokenizer(p)["input_ids"]) + len(tokenizer(c)["input_ids"])
    for p, c in zip(train_dataset["prompt"], train_dataset["completion"])
]

print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}")
for threshold in [512, 768, 1024, 1280, 1792]:
    over = sum(1 for l in lengths if l > threshold)
    print(f"  over {threshold}: {over} ({over/len(lengths)*100:.1f}%)")

Min: 193, Max: 1110, Mean: 396
  over 512: 27 (22.0%)
  over 768: 7 (5.7%)
  over 1024: 2 (1.6%)
  over 1280: 0 (0.0%)
  over 1792: 0 (0.0%)


#### DIAGNOSTIC: GLU expansion ratio (informational, for 13.3 later)

In [ ]:
def print_glu_expansion_ratio(model):
    hidden_size = model.config.hidden_size
    intermediate_size = model.config.intermediate_size
    ratio = intermediate_size / hidden_size
    print(f"Hidden size:        {hidden_size}")
    print(f"Intermediate size:  {intermediate_size}")
    print(f"GLU expansion ratio: {ratio:.2f}x")
    return ratio

## C. Evaluation harness

Definitions only — nothing is executed here. Every function takes the model to
test as an explicit argument, so these cells never need to be re-run in a
particular order relative to model loading, and the diagnostics in section G can
call them safely at any point.

In [ ]:
def generate_tool_call(model, tokenizer, example, max_new_tokens=200):
    tools = json.loads(example["tools"])
    messages = [{"role": "user", "content": example["query"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # Qwen3 ships sampling defaults in its generation_config; passing
            # them as None with do_sample=False silences the "generation flags
            # are not valid" warning instead of ignoring them at every call.
            temperature=None, top_p=None, top_k=None,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def parse_tool_call(generated_text):
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

### Tolerant matching

On top of basic number and case normalization, two functional equivalences count
as correct — both decided after auditing real mismatches:

- a city name that is a superset of the ground truth (`"Sydney, Australia"` vs
  `"Sydney"`);
- coordinates within 0.05 degrees (~5 km), well within the variation between two
  sources for "the coordinates of a city".

Substring equivalence is restricted to values that contain letters. Cleaning
strips `.` and `,`, which would otherwise let `"54.239.26.128"` match
`"154.239.26.128"` — numeric closeness is `is_close_coordinate`'s job.

In [ ]:
def is_substring_equivalent(a, b):
    a_clean = re.sub(r"[^\w\s]", "", str(a).lower()).strip()
    b_clean = re.sub(r"[^\w\s]", "", str(b).lower()).strip()
    if not a_clean or not b_clean:
        return False
    # Free-text values only. If either side has no letters after cleaning, bail
    # out: digit-substring collisions would otherwise produce false positives.
    if not re.search(r"[a-z]", a_clean) or not re.search(r"[a-z]", b_clean):
        return False
    # Require a minimum length so short strings don't match by sheer chance
    # (e.g. "i" being a substring of "imperial").
    if len(a_clean) < 3 or len(b_clean) < 3:
        return a_clean == b_clean
    return a_clean in b_clean or b_clean in a_clean


def is_close_coordinate(key, predicted, expected, tolerance_degrees=0.05):
    if not any(k in key.lower() for k in ["lat", "lon", "lng"]):
        return False
    try:
        return abs(float(predicted) - float(expected)) <= tolerance_degrees
    except (TypeError, ValueError):
        return False


def value_matches(key, predicted, expected, tolerant):
    if tolerant:
        try:
            if float(predicted) == float(expected):
                return True
        except (TypeError, ValueError):
            pass
        if isinstance(predicted, str) and isinstance(expected, str):
            if predicted.strip().lower() == expected.strip().lower():
                return True
            # Whitespace inside a purely numeric value is formatting, not
            # content: "-33.8688, 151.2093" and "-33.8688,151.2093" are the same
            # coordinate pair. Restricted to values with no letters so that API
            # naming conventions ("Park City" vs "parkcity") still count as a
            # miss — that is something fine-tuning should learn.
            if not re.search(r"[a-z]", (predicted + expected).lower()):
                if re.sub(r"\s+", "", predicted) == re.sub(r"\s+", "", expected):
                    return True
            if is_substring_equivalent(predicted, expected):
                return True
        if is_close_coordinate(key, predicted, expected):
            return True
    return predicted == expected


def call_matches(predicted, ground_truth, tolerant):
    if predicted is None or predicted.get("name") != ground_truth["name"]:
        return False
    predicted_args = predicted.get("arguments", {})
    expected_args = ground_truth["arguments"]
    # In tolerant mode a missing key still fails, but an extra key does not.
    if not tolerant and set(predicted_args.keys()) != set(expected_args.keys()):
        return False
    return all(
        key in predicted_args and value_matches(key, predicted_args[key], expected_value, tolerant)
        for key, expected_value in expected_args.items()
    )

In [ ]:
def evaluate_model(model, tokenizer, examples, verbose=True):
    """Evaluate and keep what was generated.

    Each result carries the raw output and the parsed call, so downstream
    analysis reads the very same generations these metrics were computed from —
    no regeneration, and no assumption that a second pass would produce
    identical text.
    """
    results = []
    for example in examples:
        ground_truth = json.loads(example["answers"])[0]
        generated_text = generate_tool_call(model, tokenizer, example)
        predicted = parse_tool_call(generated_text)
        results.append({
            "generated_text": generated_text,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "valid_json": predicted is not None,
            "exact_match": call_matches(predicted, ground_truth, tolerant=False),
            "tolerant_match": call_matches(predicted, ground_truth, tolerant=True),
        })

    n = len(results)
    if verbose:
        print(f"  Valid JSON:      {sum(r['valid_json'] for r in results) / n:.1%}")
        print(f"  Exact match:     {sum(r['exact_match'] for r in results) / n:.1%}")
        print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in results) / n:.1%}")
    return results


def breakdown_mismatches(results, verbose=True):
    no_call = wrong_function = wrong_args = 0
    for result in results:
        if result["tolerant_match"]:
            continue
        predicted = result["predicted"]
        if predicted is None:
            no_call += 1
        elif predicted.get("name") != result["ground_truth"]["name"]:
            wrong_function += 1
        else:
            wrong_args += 1

    if verbose:
        print(f"  No tool_call emitted:        {no_call}")
        print(f"  Wrong function selected:     {wrong_function}")
        print(f"  Right function, wrong args:  {wrong_args}")
    return {"no_call": no_call, "wrong_function": wrong_function, "wrong_args": wrong_args}

## D. Get the specialist model

`TRAIN_MODEL` selects between two paths, but both end with one variable:
`specialist_model`, ready to evaluate.

When training, the model is loaded with fp16 weights, trained with LoRA under
fp16 AMP, and then the adapter is merged into the trained model. When training
is skipped, the merged specialist is loaded from `HF_REPO_ID`. In either case,
the model is evaluated without quantization, and the adapter is saved before a
newly trained model is merged.

With no quantization, merging is a plain weight addition on the model that was
just trained: there is no dequantize/add/requantize round trip.

In [ ]:
if TRAIN_MODEL:
    # --- D1: base model in fp32, no quantization ---
    # Qwen3-1.7B in fp32 is ~6.8 GB of weights; LoRA only adds optimizer state
    # for the adapters, so this fits on a 16 GB T4 with gradient checkpointing.
    print(f"Loading {MODEL_NAME} (fp32, for LoRA training)...")
    train_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="auto", dtype=torch.float16, low_cpu_mem_usage=True,
    )
    train_model.config.use_cache = False
    train_model.generation_config.pad_token_id = tokenizer.pad_token_id
    train_model.gradient_checkpointing_enable()
    train_model.enable_input_require_grads()  # required for checkpointing + PEFT

    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES,
    )
    train_model = get_peft_model(train_model, lora_config)
    train_model.print_trainable_parameters()

    # --- D2: training arguments ---
    training_args = SFTConfig(
        output_dir="./lora_output",
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=3,
        fp16=True,          # AMP + GradScaler; works on T4 and L4 alike
        bf16=False,         # not available on Turing (T4)
        logging_steps=10,
        save_strategy="no",
        max_length=MAX_SEQ_LENGTH,
        completion_only_loss=True,
        gradient_checkpointing=True,
        report_to="none",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
    )
    trainer = SFTTrainer(model=train_model, train_dataset=train_dataset, args=training_args)

    # --- D3: verify the loss mask BEFORE spending time on training ---
    batch = trainer.get_train_dataloader().__iter__().__next__()
    labels = batch["labels"][0]
    n_masked = (labels == -100).sum().item()
    n_total = labels.numel()
    print(f"Masked tokens (label = -100): {n_masked} / {n_total} ({n_masked/n_total:.1%})")
    assert n_masked > 0, "completion_only_loss did NOT apply — stop and investigate."
    del batch, labels

    # --- D4: train ---
    print("Starting LoRA training...")
    trainer.train()
    print("Training complete.")

    # --- D5: save the adapter before merging (safety net) ---
    train_model.save_pretrained("./lora_adapter")
    tokenizer.save_pretrained("./lora_adapter")
    print("Adapter saved to ./lora_adapter")

    # --- D6: merge in place, then cast to fp16 for evaluation ---
    specialist_model = train_model.merge_and_unload()
    #specialist_model = specialist_model.half()
    specialist_model.config.use_cache = True
    specialist_model.gradient_checkpointing_disable()
    specialist_model.eval()

    # --- D7: free the trainer, or the VRAM is never released ---
    # del train_model alone is not enough: trainer still holds references to
    # trainer.model, trainer.model_wrapped and the optimizer state.
    del trainer, train_model
    gc.collect()
    torch.cuda.empty_cache()

    # --- D8: optional push ---
    if PUSH_TO_HUB:
        from huggingface_hub import login
        login()
        specialist_model.push_to_hub(HF_REPO_ID, private=True)
        tokenizer.push_to_hub(HF_REPO_ID, private=True)
        print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

else:
    print(f"TRAIN_MODEL is False — loading specialist_model from {HF_REPO_ID}...")
    specialist_model = AutoModelForCausalLM.from_pretrained(
        HF_REPO_ID, device_map="auto", dtype=torch.float16,
    )
    specialist_model.eval()

specialist_model.generation_config.pad_token_id = tokenizer.pad_token_id
print()
print("specialist_model ready (fp16, merged).")

Loading Qwen/Qwen3-0.6B (fp32, for LoRA training)...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 10,092,544 || all params: 761,724,928 || trainable%: 1.3250


Adding EOS to train dataset:   0%|          | 0/123 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9g1_buv8/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed to https://huggingface.co/oopere/qwen3-0.6b-weather-geo-specialist

specialist_model ready (fp16, merged).


## E. Base model (reference)

A fresh copy of `MODEL_NAME` is loaded in fp16 with no fine-tuning or
quantization. Comparing it with `specialist_model` keeps model precision and
loading conditions aligned while measuring the effect of fine-tuning. The base
model is loaded on demand rather than kept alive during training, which avoids
depending on whether an earlier cell freed its memory.

In [ ]:
def load_base_model():
    gc.collect()
    torch.cuda.empty_cache()
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map="auto", dtype=torch.float16,
    )
    m.generation_config.pad_token_id = tokenizer.pad_token_id
    m.eval()
    return m


base_model = load_base_model()
print("base_model ready (fp16, no fine-tuning).")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


base_model ready (fp16, no fine-tuning).


## F. Comparison

In [ ]:
def compare_models(base_model, specialist_model, tokenizer, examples):
    print("=" * 60)
    print("BASE MODEL (no fine-tuning)")
    print("=" * 60)
    base_results = evaluate_model(base_model, tokenizer, examples)
    base_breakdown = breakdown_mismatches(base_results)

    print()
    print("=" * 60)
    print("FINE-TUNED SPECIALIST")
    print("=" * 60)
    specialist_results = evaluate_model(specialist_model, tokenizer, examples)
    specialist_breakdown = breakdown_mismatches(specialist_results)

    return {
        "base_results": base_results, "base_breakdown": base_breakdown,
        "specialist_results": specialist_results, "specialist_breakdown": specialist_breakdown,
    }


comparison = compare_models(base_model, specialist_model, tokenizer, test_examples)

BASE MODEL (no fine-tuning)
  Valid JSON:      100.0%
  Exact match:     88.9%
  Tolerant match:  88.9%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  6

FINE-TUNED SPECIALIST
  Valid JSON:      100.0%
  Exact match:     96.3%
  Tolerant match:  96.3%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  2


## G. Diagnostics on demand

Each cell recomputes what it needs from the model passed in — none of these
depend on a `_results` variable computed earlier in the notebook, so they can be
re-run safely in any order.

### G1. Overfitting check: evaluate on a TRAIN sample

In [ ]:
random.seed(RANDOM_STATE)
train_sample = random.sample(train_examples, 60)

print("specialist_model on a TRAIN sample:")
train_sample_results = evaluate_model(specialist_model, tokenizer, train_sample)
train_sample_breakdown = breakdown_mismatches(train_sample_results)

specialist_model on a TRAIN sample:
  Valid JSON:      100.0%
  Exact match:     98.3%
  Tolerant match:  98.3%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  1


### G2. Mismatch audit

Four buckets, pivoting on **exact** match rather than tolerant. Anything that
fails exact but passes tolerant is a formatting or equivalence issue, not a
modelling failure — separating those two is the point of this cell, and it only
works if the pivot is `exact_match`.

In [ ]:
def categorize_mismatches(model, tokenizer, examples):
    results = evaluate_model(model, tokenizer, examples, verbose=False)
    categories = Counter()
    details = []

    for example, result in zip(examples, results):
        predicted = result["predicted"]
        ground_truth = result["ground_truth"]

        if result["exact_match"]:
            category = "exact"
        elif result["tolerant_match"]:
            category = "tolerance_worthy"
        elif predicted is None or predicted.get("name") != ground_truth["name"]:
            category = "no_call_or_wrong_function"
        else:
            category = "wrong_arguments"

        categories[category] += 1
        details.append((example, predicted, ground_truth, category))

    print(f"Total test examples: {len(examples)}")
    for category, count in categories.most_common():
        print(f"  {category}: {count} ({count/len(examples):.1%})")

    return details

In [ ]:
mismatch_details = categorize_mismatches(specialist_model, tokenizer, test_examples)

Total test examples: 54
  exact: 52 (96.3%)
  wrong_arguments: 2 (3.7%)


In [ ]:
# Inspect the wrong-argument cases only
wrong_args = [d for d in mismatch_details if d[3] == "wrong_arguments"]
print(f"Wrong-argument cases: {len(wrong_args)}\n")

for example, predicted, ground_truth, category in wrong_args:
    print("QUERY:", example["query"])
    print("PREDICTED:", predicted)
    print("GROUND TRUTH:", ground_truth)
    print("-" * 60)

Wrong-argument cases: 2

QUERY: I'm trying to find out the ZIP code of the IP address that belongs to the Facebook's DNS server. Can you help me with that?
PREDICTED: {'name': 'get_ip_zipcode', 'arguments': {'ip': '8.8.8.8'}}
GROUND TRUTH: {'name': 'get_ip_zipcode', 'arguments': {'ip': '69.171.247.12'}}
------------------------------------------------------------
QUERY: What are the weather conditions and a 4-day forecast for Los Angeles, without air quality data, in Spanish?
PREDICTED: {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles', 'aqi': 'no', 'lang': 'es'}}
GROUND TRUTH: {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles', 'num_of_days': 4, 'aqi': 'no', 'lang': 'es'}}
------------------------------------------------------------
